# ML-09 — Validation and Research Claim Audit

This notebook audits the Week-5 Random Forest model under an honest client-grouped holdout and separates measured evidence from stronger research claims.


## 1. Two research findings and methodology questions

### Finding 1 — Content Lifecycle: Growing vs Declining
FlyRank reports that growing pages were younger on average than declining pages (185 vs 228 days), while average word count was nearly the same (1,487 vs 1,481). The comparison is observational.

**Methodology question:** Does the age difference remain after accounting for client/site grouping and other page characteristics, or could client mix, visibility, or other confounders explain part of the observed gap?

### Finding 4 — The Freshness Multiplier
FlyRank reports a 31–90 day freshness-window growth-to-decline ratio of 5.43:1 and a separate 365+ refreshed-vs-stale cohort comparison showing higher health and impressions for the refreshed cohort. The report notes that the 361+ ratio is unstable because its declining count is small.

**Methodology question:** Is the refreshed-vs-stale difference evidence of a refresh effect, or could selection effects explain part of it because pages chosen for refresh may already have stronger demand or different characteristics?

Source: FlyRank, The State of AI-Driven SEO — Data Report, April 2026.


## 2. Load the same teaching dataset used in Week 5


In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import sklearn

DATA_PATH = Path(os.environ.get('FLYRANK_DATA_PATH', 'data/raw/content_refresh_anonymized.csv'))
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    DATA_URL = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
    df = pd.read_csv(DATA_URL)
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates('content_id').reset_index(drop=True)
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
print(f'Rows used: {len(df):,}')
print(f'Clients: {df.client_id.nunique():,}')
print(f'Decline base rate: {df.is_declining_label.mean():.2%}')
print(f'Python: {sys.version.split()[0]}')
print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')
print(f'scikit-learn: {sklearn.__version__}')


## 3. Feature and leakage audit
The label comes from trend_direction. trend_direction, trend_pct, the constructed target, content_id, and client_id are excluded from predictive features. client_id is used only for grouping.


In [ ]:
numeric_features = ['search_volume','cpc','word_count','char_count','impressions_90d','clicks_90d','sessions_90d','ai_sessions_90d','days_with_impressions_90d','days_with_sessions_90d','content_age_days','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct']
categorical_features = ['competition','content_type','main_intent','competition_level','age_tier','freshness_tier','word_count_tier','impression_tier','position_tier']
excluded = ['trend_direction','trend_pct','is_declining_label','content_id','client_id']
features = [c for c in numeric_features + categorical_features if c in df.columns]
assert not any(c in features for c in excluded)
print('Leakage audit: PASS')
print('Excluded:', excluded)
print('Model features:', len(features))


## 4. Naive row split versus honest client-grouped split
The naive split randomly separates rows, so the same client can appear in both train and test. The honest split keeps every client's rows together and is the primary validation estimate.


In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def build_model():
    nums = [c for c in numeric_features if c in df.columns]
    cats = [c for c in categorical_features if c in df.columns]
    prep = ColumnTransformer([('num', SimpleImputer(strategy='median'), nums), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cats)])
    return Pipeline([('prep', prep), ('rf', RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced_subsample', random_state=42, n_jobs=-1))])

X = df[features]
y = df['is_declining_label']
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
naive_model = build_model().fit(Xn_tr, yn_tr)
naive_score = naive_model.predict_proba(Xn_te)[:, 1]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
gtr, gte = next(gss.split(X, y, groups=df['client_id']))
Xg_tr, Xg_te = X.iloc[gtr], X.iloc[gte]
yg_tr, yg_te = y.iloc[gtr], y.iloc[gte]
group_model = build_model().fit(Xg_tr, yg_tr)
group_score = group_model.predict_proba(Xg_te)[:, 1]

naive_train_clients = set(df.iloc[Xn_tr.index]['client_id'])
naive_test_clients = set(df.iloc[Xn_te.index]['client_id'])
group_train_clients = set(df.iloc[gtr]['client_id'])
group_test_clients = set(df.iloc[gte]['client_id'])

comparison = pd.DataFrame({
    'validation': ['Naive random row split', 'Honest client-grouped split'],
    'ROC AUC': [roc_auc_score(yn_te, naive_score), roc_auc_score(yg_te, group_score)],
    'Average Precision': [average_precision_score(yn_te, naive_score), average_precision_score(yg_te, group_score)],
    'train rows': [len(Xn_tr), len(Xg_tr)],
    'test rows': [len(Xn_te), len(Xg_te)],
    'train clients': [len(naive_train_clients), len(group_train_clients)],
    'test clients': [len(naive_test_clients), len(group_test_clients)],
    'client overlap': [len(naive_train_clients & naive_test_clients), len(group_train_clients & group_test_clients)]
}).round(3)
display(comparison)


The grouped split is the primary estimate because it evaluates on clients not used for training. Any gap from the naive split is a validation-design effect, not a claim about real-world improvement or decline.


## 5. Error examples on the honest split
These are concrete evaluation cases, not causal explanations.


In [ ]:
group_test = df.iloc[gte][['content_id','client_id','trend_direction','content_age_days','days_since_last_update','impressions_90d','avg_position','word_count']].copy()
group_test['actual'] = yg_te.values
group_test['score'] = group_score
group_test['prediction'] = (group_score >= 0.50).astype(int)
false_positives = group_test[(group_test.actual == 0) & (group_test.prediction == 1)].sort_values('score', ascending=False).head(3)
false_negatives = group_test[(group_test.actual == 1) & (group_test.prediction == 0)].sort_values('score', ascending=True).head(3)
print('False positives:')
display(false_positives)
print('False negatives:')
display(false_negatives)


## 6. Research claim audit and safer rewrites

| Original evidence | Audit question | Safer claim |
|---|---|---|
| Growing pages were younger on average than declining pages. | This is an aggregate observational comparison. Does client mix or other page characteristics contribute? | **Observed in this dataset:** growing pages were younger on average than declining pages. The comparison does not by itself establish that age causes growth. |
| 31–90 day freshness window had a 5.43:1 growth-to-decline ratio. | Are the groups comparable, and how stable are the counts? | **Measured association:** pages in the 31–90 day freshness window had a 5.43:1 growth-to-decline ratio in the study window. This should not be read as a causal freshness effect. |
| 365+ refreshed pages showed higher health and impressions than a stale cohort. | Could selection effects explain part of the difference? | **Observed cohort difference:** the refreshed 365+ cohort had higher reported health and impressions than the comparison stale cohort; selection effects cannot be ruled out from this comparison alone. |

Use observed, measured, associated, descriptive, and decision-support language unless the design supports causal claims.


## 7. Self-check


In [ ]:
checks = {
    'two research findings named': True,
    'methodology question for each finding': True,
    'naive versus honest split shown': True,
    'honest split has zero client overlap': len(group_train_clients & group_test_clients) == 0,
    'target-derived fields excluded': not any(c in features for c in ['trend_direction','trend_pct','is_declining_label']),
    'error examples included': len(false_positives) > 0 and len(false_negatives) > 0,
    'claims rewritten with evidence-safe language': True
}
display(pd.Series(checks, name='pass'))
assert all(checks.values())
print('ML-09 self-check: PASS')
